# 05 - Weighted conformal prediction (Tibshirani et al., 2019)

Reweight each calibration score by how much that bright object resembles the faint target population, estimated with a domain classifier (bright=0, faint=1) on the same features. This is the correction to the undercoverage shown in notebook 04.

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

cal = pd.read_parquet("../data/processed/cal_with_preds.parquet")
target = pd.read_parquet("../data/processed/target_with_preds.parquet")
FEATURE_COLS = json.load(open("../data/processed/feature_cols.json"))
ALPHA = 0.1

## Domain classifier: calibration (0) vs. target (1)

In [2]:
X_dom = pd.concat([cal[FEATURE_COLS], target[FEATURE_COLS]])
y_dom = np.concatenate([np.zeros(len(cal)), np.ones(len(target))])

clf = LogisticRegression(max_iter=2000)
clf.fit(X_dom, y_dom)

# AUC near 1.0 is a warning sign: it means bright/faint are almost perfectly separable,
# so weights will be extreme and the correction unreliable in the most shifted region
auc = roc_auc_score(y_dom, clf.predict_proba(X_dom)[:, 1])
auc

0.99992392608997

In [3]:
n0, n1 = len(cal), len(target)

def odds_weight(X):
    pi = clf.predict_proba(X)[:, 1]
    pi = np.clip(pi, 1e-3, 1 - 1e-3)
    return (n0 / n1) * pi / (1 - pi)

w_cal = odds_weight(cal[FEATURE_COLS])
w_target = odds_weight(target[FEATURE_COLS])

# clip extreme weights at the 99th percentile of the calibration weights
clip_val = np.quantile(w_cal, 0.99)
w_cal = np.clip(w_cal, None, clip_val)
w_target = np.clip(w_target, None, clip_val)

# effective sample size: collapses if a few calibration points dominate the weights
n_eff = (w_cal.sum() ** 2) / (w_cal ** 2).sum()
print(f"clip value: {clip_val:.2f}  n_eff: {n_eff:.1f} (of {n0} calibration points)")

clip value: 0.44  n_eff: 48.3 (of 596 calibration points)


## Weighted quantile per target point (augmented with a +inf point mass for the unseen test weight)

In [4]:
def weighted_conformal(y_cal, yhat_cal, w_cal, yhat_test, w_test, alpha=0.1):
    scores = np.abs(y_cal - yhat_cal) / (1 + yhat_cal)
    order = np.argsort(scores)
    scores_sorted, w_sorted = scores[order], w_cal[order]
    q_hats = np.empty(len(w_test))
    for i, w_t in enumerate(w_test):
        weights_aug = np.append(w_sorted, w_t)
        weights_aug /= weights_aug.sum()
        cum = np.cumsum(weights_aug)
        idx = min(np.searchsorted(cum, 1 - alpha), len(scores_sorted) - 1)
        q_hats[i] = scores_sorted[idx]
    return q_hats

q_hats = weighted_conformal(
    cal["z_spec"].values, cal["z_pred"].values, w_cal,
    target["z_pred"].values, w_target, alpha=ALPHA,
)
target["z_lower_w"] = target["z_pred"] - q_hats * (1 + target["z_pred"])
target["z_upper_w"] = target["z_pred"] + q_hats * (1 + target["z_pred"])
target["covered_w"] = (target["z_spec"] >= target["z_lower_w"]) & (target["z_spec"] <= target["z_upper_w"])

coverage_w = target["covered_w"].mean()
width_w = (target["z_upper_w"] - target["z_lower_w"]).mean()
print(f"nominal coverage: {1 - ALPHA:.2f}")
print(f"empirical coverage, weighted: {coverage_w:.3f}")
print(f"mean interval width, weighted: {width_w:.3f}")

nominal coverage: 0.90
empirical coverage, weighted: 0.735
mean interval width, weighted: 1.220


## Coverage by magnitude bin, weighted -- does the correction hold up in the faintest bin?

In [5]:
target["mag_bin"] = pd.qcut(target["mag_F160W"], 4).astype(str)
target.groupby("mag_bin")["covered_w"].agg(["mean", "count"])

,mean,count
mag_bin,,
"(21.575, 22.053]",0.857143,497
"(22.053, 22.515]",0.826613,496
"(22.515, 23.027]",0.778226,496
"(23.027, 26.491]",0.477823,496


In [6]:
target.to_parquet("../data/processed/target_weighted.parquet", index=False)
json.dump(
    {"auc": float(auc), "n_eff": float(n_eff), "clip_val": float(clip_val),
     "coverage_weighted": float(coverage_w), "width_weighted": float(width_w)},
    open("../data/processed/weighted_metrics.json", "w"),
)